# Simulación del juego Go

## Introducción

En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en el juego Go. Go es un juego de estrategia para dos jugadores en el que se colocan fichas en un tablero con el objetivo de controlar el mayor territorio al final de la partida.

Reglas:
 - El juego se juega en un tablero cuadrado con un número impar de filas y columnas entre 5 y 19 (por defecto 9x9).
 - Los jugadores se turnan para colocar fichas de su color (representadas por 1 o 2) en las intersecciones vacías del tablero.
 - Una jugada válida debe respetar las reglas de no suicidio y Superko situacional:
    - Regla de no suicidio: Un jugador no puede colocar una ficha que haga que su propio grupo quede sin libertades inmediatamente, a menos que esa jugada capture fichas enemigas que devuelvan libertades al grupo.
    - Regla de Superko situacional: Un jugador no puede realizar una jugada que haga que el tablero y el turno actual coincidan exactamente con una situación previa en la misma partida (para evitar ciclos infinitos).
 - Se llaman libertades a las casillas vacías adyacentes a una cierta ficha o grupo de fichas del mismo color.
 - Si al colocar una ficha se rodean grupos enemigos sin libertades, estas fichas se capturan y se retiran del tablero.
 - Un jugador puede pasar su turno, y si ambos jugadores pasan consecutivamente, el juego termina.
 - La puntuación final de cada jugador se calcula sumando:
    - El número de fichas en el tablero de dicho jugador.
    - Las fichas capturadas al oponente.
    - El territorio rodeado exclusivamente por fichas del color de dicho jugador.
    - La compensación Komi de 6.5 puntos para el jugador que no comenzó.
 - Gana el jugador con más puntos al final de la partida.

 En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en Go.

In [2]:
import sys
import os
import pandas as pd
import random
import numpy as np

# Fijar la semilla
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

# Agregar los directorios al path para poder importar los módulos
sys.path.append(os.path.abspath("../games"))
sys.path.append(os.path.abspath("../methods"))
sys.path.append(os.path.abspath("../selection"))
sys.path.append(os.path.abspath("../graphviz"))

# Importar la clase GoGame y las funciones de los métodos
from go import GoGame
from monte_carlo import MCPlay, MCAgent
from monte_carlo_tree_search import MCTS, MCTSPlay, Node, MCTSAgent
from epsilon_greedy import EpsilonGreedy
from softmax import Softmax
from adaptive_softmax import AdaptiveSoftmax
from ucb1 import UCB1
from ucb2 import UCB2
from gradiente_preferencias import GradienteDePreferencias
from MCTS_graphviz import generate_tree_MCTS

In [ ]:
import sys
import pandas
import numpy
import graphviz

print("sys:", sys.version)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("graphviz:", graphviz.__version__)

sys: 3.8.5 (tags/v3.8.5:580fbb0, Jul 20 2020, 15:57:54) [MSC v.1924 64 bit (AMD64)]
pandas: 2.0.3
numpy: 1.23.5
graphviz: 0.20.3


## Implementación de la clase `GoGame`

En esta sección, se explica cómo ha sido implementada la clase `GoGame` que modela el juego. Para ello, a continuación se presenta una breve descripción de cada uno de los métodos que la componen:

 - `__init__(self, rows=9, cols=9, starting_player=1)`: Se utiliza para inicializar el juego. Inicializa el tablero de tamaño `rows` x `cols` con casillas vacías (0), además de un historial de movimientos, un contador de pases y los contadores de puntuación para cada jugador. Además, establece el jugador inicial (1 o -1) y asigna la compesación Komi al jugador que no inicia la partida para equilibrar la ventaja del otro jugador. Sus parámetros incluyen:
   - `rows`: número de filas del tablero (de 5 a 19, debe ser impar).
   - `cols`: número de columnas del tablero (de 5 a 19, debe ser impar).
   - `starting_player`: el jugador que empieza el juego (1 o -1).

 - `comprobar_limites(self, row, col)`: Este método verifica si la posición `(row, col)` está dentro de los límites del tablero.

 - `find_group(self, board, row, col, turn)`: Determina qué fichas forman parte del grupo contectado a la ficha en `(row, col)` del jugador `turn`. Este grupo se obtiene explorando en 4 direcciones (arriba, abajo, derecha, izquierda).

 - `has_liberties(self, board, group)`: Comprueba si un grupo de fichas tiene libertades, es decir, si existe al menos una casilla vacía adyacente a alguna ficha del grupo. Si no tiene libertades, entonces el grupo está capturado.

 - `eliminate_enemy_groups(self, board, row, col)`: Identifica los grupos enemigos adyacentes que no tengan libertades después de colocar una ficha en `(row, col)` y los elimina. Las fichas eliminadas se suman a la puntuación del jugador que realiza la captura.

 - `valid_actions(self)`: Este método devuelve las acciones válidas en el estado actual del tablero. Una jugada es válida si la casilla está vacía, no incumple la regla del suicidio (esto es, que el grupo resultante al realizar la jugada tenga al menos una libertad y, por tanto, no quede capturado), y cumple la regla de Situational Superko (el tablero resultante no repite un estado anterior).

 - `action(self, coordinates)`: Este método simula un turno del juego en la posición dada por las coordenadas `coordinates`, de la forma `(row, col)`. Si se indica una posición válida, coloca una ficha del jugador actual, elimina los grupos enemigos que se queden sin libertades, y genera un nuevo estado del juego cambiando el turno al jugador contrario. Si no se indican unas coordenadas, se pasa el turno al siguiente jugador.

 - `terminal(self)`: Verifica si el juego ha terminado, lo cual sucede cuando ambos jugadores pasan turno de forma consecutiva.

 - `winner(self)`: Devuelve quién es el ganador de la partida. Si el juego ha terminado, devuelve el número del jugador ganador (1 o -1) o un 0 si ha habido empate, pero si no ha terminado devuelve 0. La puntuación final de cada jugador se calcula sumando:
    - El número de fichas en el tablero de dicho jugador.
    - Las fichas capturadas al oponente.
    - El territorio rodeado exclusivamente por fichas del color de dicho jugador.
    - La compensación Komi de 6.5 puntos para el jugador que no comenzó.

 - `draw(self)`: Este método dibuja el estado actual del juego, imprimiendo el tablero en formato de matriz, donde cada celda representa una ficha (1, 2, o 0 si está vacía).

## Simulaciones

En esta sección, se realizan simulaciones con distintas estrategias para la toma de decisiones en el juego. Comenzamos creando una instancia del juego `GoGame`, y visualizamos su estado inicial. Por defecto, se juega con un tablero con 9 filas y 9 columnas. Sin embargo, estos parámetros se pueden modificar con `rows` y `cols`.

In [2]:
# Crear una instancia del juego Go con los valores predeterminados
game = GoGame()

# Mostrar el estado inicial del juego
game.draw()

[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]


A continuación, se presentan los distintos métodos utilizados para realizar las simulaciones.

## Monte-Carlo

En primer lugar, se implementa el método de Monte-Carlo para estimar las mejores jugadas. 

In [2]:
# Crear una instancia del juego Go
game = GoGame(rows=5, cols=5, starting_player=-1)

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 20

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = eval(input("¿En qué casilla (fila, columna) quieres colocar tu ficha? (None para pasar de turno): "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue  # Volver a pedir jugada

    else:  # Turno de la IA (Monte-Carlo)
        print("\nTurno de la computadora...")
        game = MCPlay(game, num_simulations)

    game.draw()  # Mostrar el estado después del turno

# Anunciar el ganador
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0]
[0, 0, 0, 2, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0]
[0, 0, 1, 2, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0]
[2, 0, 1, 2, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 1, 0]
[2, 0, 1, 2, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 1, 0]
[2, 0, 1, 2, 0]
[0, 0, 2, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 1, 0]
[2, 0, 1, 2, 0]
[0, 0, 2, 1, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 1, 0]
[2, 2, 1, 2, 0]
[0, 0, 2, 1, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 1, 0]
[2, 2, 1, 0, 1]
[0, 0, 2, 1, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 1, 0]
[2, 2, 1, 0, 1]
[0, 0, 2, 1, 0]
[0, 0, 0, 2, 0]
[0, 0, 0, 0, 0]

Tu turn

### Monte-Carlo Tree Search

En esta sección, se utiliza el método de Monte-Carlo Tree Search para estimar las mejores jugadas en el juego Go. Este algoritmo está compuesto por distintas fases, que son:

 1. **Selección:** En esta fase, se parte del nodo raíz y se desciende por el árbol seleccionando nodos hijos sucesivamente según una estrategia. Esto continúa hasta llegar a un nodo hoja, es decir, un nodo que tiene al menos un hijo potencial al que no se le ha aplicado ninguna simulación todavía. Entre las estrategias de selección encontramos: $\epsilon$-greedy, Softmax, Adaptive Softmax, UCB1, UCB2 y Gradiente de Preferencias.

 2. **Expansión:** A partir del nodo hoja seleccionado, se genera un nuevo nodo hijo, es decir, se aplica un movimiento válido que aún no ha sido explorado desde el nodo hoja.

 3. **Simulación:** Desde el nodo hijo recién creado, se completa una jugada aleatoria hasta alcanzar un estado terminal (por ejemplo, una victoria, derrota o empate). Mediante esta simulación, se puede estimar el resultado potencial de seguir esa línea de decisión.

 4. **Retropropagación:** Los resultados de la simulación se propagan hacia atrás, usándose para actualizar las estadísticas de los nodos, que son recorridos desde el nodo expandido hasta el nodo raíz. Esto permite que en futuras decisiones se refuercen las rutas más prometedoras y se descarten las menos efectivas.

In [ ]:
# Crear una instancia del juego Othello
game = GoGame()

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 20

# Escoger el método de selección
selection_algorithm_class = GradienteDePreferencias

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = eval(input("¿En qué casilla (fila, columna) quieres colocar tu ficha? (None para pasar de turno): "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue

    else:  # Turno de la IA con MCTS
        print("\nTurno de la computadora...")
        game = MCTSPlay(game, num_simulations, selection_algorithm_class)

    game.draw()

# Resultado
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

A continuación, se utiliza `graphviz` como herramienta de depuración, generando un diagrama con el árbol de posiciones siguientes a partir de una determinada posición inicial, dada por un nodo raíz. Cada nodo del árbol representa un estado del juego, el cual se alcanza mediante una secuencia de movimientos, y contiene información como el turno del jugador, el número de visitas y la recompensa de cada jugador. Las aristas indican las acciones tomadas para pasar de un estado al siguiente. Se puede especificar la profundidad máxima del árbol generado.

In [ ]:
# Crear estado inicial del juego
initial_state = GoGame()

# Visualizar el árbol de MCTS
root_node = Node(initial_state, None)
mcts = MCTS(root_node, EpsilonGreedy, simulations=50, epsilon=0.2)
mcts.run()
id_to_node = generate_tree_MCTS(root_node, max_depth=2, filename="depuration_trees/arbol_go", format="pdf")

Los nodos en el árbol aparecen identificados mediante un ID. Para poder visualizar cuál es el estado de cada nodo, la función `generate_tree_MCTS` devuelve un diccionario que asocia cada ID al nodo correspondiente.

In [ ]:
node = id_to_node['nodo1']

node.state.draw()

## Simulaciones comparativas

In [3]:
def play_comparative_game(agent1, agent2, rows=5, cols=5, starting_player=1):
    game = GoGame(rows=rows, cols=cols, starting_player=starting_player)
    agents = {1: agent1, -1: agent2}

    while not game.terminal():
        game = agents[game.turn].move(game)

    return game.winner()

In [4]:
def run_simulations(agent_1, agent_2, rows=5, cols=5, n_games=100):
    results = []

    for i in range(n_games):
        # Alternar jugador inicial
        starting_player = 1 if i % 2 == 0 else -1

        # Asignar agentes según quién empieza
        if starting_player == 1:
            winner = play_comparative_game(agent_1, agent_2, rows, cols, starting_player)
        else:
            winner = play_comparative_game(agent_2, agent_1, rows, cols, starting_player)
            # Invertir perspectiva
            winner *= -1

        results.append(winner)

    df = pd.DataFrame(results, columns=["winner"])
    win_rate = (df["winner"] == 1).mean()
    print(f"% de partidas ganadas por el agente 1: {win_rate:.2%} ({df['winner'].value_counts().to_dict()})")
    return df

In [42]:
agent_mc = MCAgent(10)
agent_mcts_eps = MCTSAgent(10, EpsilonGreedy, epsilon = 0.2)
agent_mcts_UCB1 = MCTSAgent(10, UCB1, c=1)
agent_mcts_UCB2 = MCTSAgent(10, UCB2, alpha=0.5)
agent_mcts_soft = MCTSAgent(10, Softmax, tau = 1)
agent_mcts_adapsoft = MCTSAgent(10, AdaptiveSoftmax, tau_0 = 1, alpha = 0.5)
agent_mcts_grad = MCTSAgent(10, GradienteDePreferencias, alpha = 0.2)

In [5]:
agent_mc = MCAgent(100)
agent_mcts_eps = MCTSAgent(100, EpsilonGreedy, epsilon = 0.2)
agent_mcts_UCB1 = MCTSAgent(100, UCB1, c=1)
agent_mcts_UCB2 = MCTSAgent(100, UCB2, alpha=0.5)
agent_mcts_soft = MCTSAgent(100, Softmax, tau = 1)
agent_mcts_adapsoft = MCTSAgent(100, AdaptiveSoftmax, tau_0 = 1, alpha = 0.5)
agent_mcts_grad = MCTSAgent(100, GradienteDePreferencias, alpha = 0.2)

### Monte Carlo vs MCTS con $\epsilon$-greedy

In [24]:
df_mc_mcts_eps = run_simulations(agent_mc, agent_mcts_eps, n_games=10)

% de partidas ganadas por el agente 1: 90.00% ({1: 9, -1: 1})


In [5]:
df_mc_mcts_eps = run_simulations(agent_mc, agent_mcts_eps, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


$87m$ $34.6s$

### Monte Carlo vs MCTS con UCB1

In [43]:
df_mc_mcts_UCB1 = run_simulations(agent_mc, agent_mcts_UCB1, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


In [6]:
df_mc_mcts_UCB1 = run_simulations(agent_mc, agent_mcts_UCB1, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


$115m$ $9.0s$

### Monte Carlo vs MCTS con UCB2

In [44]:
df_mc_mcts_UCB2 = run_simulations(agent_mc, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


In [7]:
df_mc_mcts_UCB2 = run_simulations(agent_mc, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


$99m$ $25.4s$

### Monte Carlo vs MCTS con Softmax

In [45]:
df_mc_mcts_soft = run_simulations(agent_mc, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


In [8]:
df_mc_mcts_soft = run_simulations(agent_mc, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


### Monte Carlo vs MCTS con Softmax Adaptativo

In [46]:
df_mc_mcts_adapsoft = run_simulations(agent_mc, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


In [9]:
df_mc_mcts_adapsoft = run_simulations(agent_mc, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 100.00% ({1: 10})


### Monte Carlo vs MCTS con Gradiente de Preferencias

In [ ]:
df_mc_mcts_grad = run_simulations(agent_mc, agent_mcts_grad, n_games=10)

### MCTS con $\epsilon$-greedy vs MCTS con UCB1

In [47]:
df_mcts_eps_UCB1 = run_simulations(agent_mcts_eps, agent_mcts_UCB1, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [7]:
df_mcts_eps_UCB1 = run_simulations(agent_mcts_eps, agent_mcts_UCB1, n_games=10)

% de partidas ganadas por el agente 1: 80.00% ({1: 8, -1: 2})


### MCTS con $\epsilon$-greedy vs MCTS con UCB2

In [48]:
df_mcts_eps_UCB2 = run_simulations(agent_mcts_eps, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [14]:
df_mcts_eps_UCB2 = run_simulations(agent_mcts_eps, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 70.00% ({1: 7, -1: 3})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax

In [49]:
df_mcts_eps_soft = run_simulations(agent_mcts_eps, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [15]:
df_mcts_eps_soft = run_simulations(agent_mcts_eps, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 70.00% ({1: 7, -1: 3})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax Adaptativo

In [50]:
df_mcts_eps_adapsoft = run_simulations(agent_mcts_eps, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [16]:
df_mcts_eps_adapsoft = run_simulations(agent_mcts_eps, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 50.00% ({-1: 5, 1: 5})


### MCTS con $\epsilon$-greedy vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_eps_grad = run_simulations(agent_mcts_eps, agent_mcts_grad, n_games=10)

### MCTS con UCB1 vs MCTS con UCB2

In [51]:
df_mcts_UCB1_UCB2 = run_simulations(agent_mcts_UCB1, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [17]:
df_mcts_UCB1_UCB2 = run_simulations(agent_mcts_UCB1, agent_mcts_UCB2, n_games=10)

% de partidas ganadas por el agente 1: 70.00% ({1: 7, -1: 3})


### MCTS con UCB1 vs MCTS con Softmax

In [52]:
df_mcts_UCB1_soft = run_simulations(agent_mcts_UCB1, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [18]:
df_mcts_UCB1_soft = run_simulations(agent_mcts_UCB1, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 70.00% ({1: 7, -1: 3})


### MCTS con UCB1 vs MCTS con Softmax Adaptativo

In [53]:
df_mcts_UCB1_adapsoft = run_simulations(agent_mcts_UCB1, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [19]:
df_mcts_UCB1_adapsoft = run_simulations(agent_mcts_UCB1, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 80.00% ({1: 8, -1: 2})


### MCTS con UCB1 vs MCTS con Gradiente de Preferencias

In [54]:
df_mcts_UCB1_grad = run_simulations(agent_mcts_UCB1, agent_mcts_grad, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


### MCTS con UCB2 vs MCTS con Softmax

In [55]:
df_mcts_UCB2_soft = run_simulations(agent_mcts_UCB2, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [20]:
df_mcts_UCB2_soft = run_simulations(agent_mcts_UCB2, agent_mcts_soft, n_games=10)

% de partidas ganadas por el agente 1: 80.00% ({1: 8, -1: 2})


### MCTS con UCB2 vs MCTS con Softmax Adaptativo

In [56]:
df_mcts_UCB2_adapsoft = run_simulations(agent_mcts_UCB2, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [21]:
df_mcts_UCB2_adapsoft = run_simulations(agent_mcts_UCB2, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 70.00% ({1: 7, -1: 3})


### MCTS con UCB2 vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_UCB2, agent_mcts_grad, n_games=10)

### MCTS con Softmax vs MCTS con Softmax Adaptativo

In [57]:
df_mcts_soft_adapsoft = run_simulations(agent_mcts_soft, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 0.00% ({-1: 10})


In [6]:
df_mcts_soft_adapsoft = run_simulations(agent_mcts_soft, agent_mcts_adapsoft, n_games=10)

% de partidas ganadas por el agente 1: 50.00% ({1: 5, -1: 5})


$12m$ $16.8s$

### MCTS con Softmax vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_soft, agent_mcts_grad, n_games=10)